In [1]:

%pip install -q langchain langchain-community langchain-text-splitters langchain-openai langchain-chroma pypdf gradio


In [2]:

import langchain
import gradio as gr
import pypdf

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

print("All required project libraries imported successfully!")
print("LangChain:", langchain.__version__)
print("Gradio:", gr.__version__)


/tmp/ipykernel_29644/622664025.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


All required project libraries imported successfully!
LangChain: 1.4.3
Gradio: 6.29.0


In [3]:
from google.colab import files
uploaded = files.upload()

Saving the_nestle_hr_policy_pdf_2012.pdf to the_nestle_hr_policy_pdf_2012 (3).pdf


In [4]:

from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader

pdf_files = list(Path("/content").glob("*.pdf"))

if not pdf_files:
    raise FileNotFoundError("Please upload your PDF first.")

pdf_path = pdf_files[0]

loader = PyPDFLoader(str(pdf_path))
documents = loader.load()

print("PDF loaded successfully!")
print("File:", pdf_path.name)
print("Total pages:", len(documents))


PDF loaded successfully!
File: the_nestle_hr_policy_pdf_2012.pdf
Total pages: 8


In [5]:

for i, doc in enumerate(documents[:2]):
    print(f"\n--- Page {i + 1} ---")
    print(doc.page_content[:1500])



--- Page 1 ---
Policy
Mandatory
September  2012
The Nestlé  
Human Resources Policy

--- Page 2 ---
Policy
Mandatory
September  20
12
Issuing departement
Hum
an Resources
T arget audience 
All
 employees
Approver
Executive Board, Nestlé S.A.
Repository
All Nestlé Principles and Policies, Standards and  
Guidelines can be found in the Centre online repository at:  
http://intranet.nestle.com/nestledocs
Copyright and confidentiality
Al
l rights belong to Nestec Ltd., Vevey, Switzerland.
© 2012, Nestec Ltd.
Design
Nestec Ltd., Corporate Identity & Design,  
Vevey, Switzerland
Production
brain’print GmbH, Switzerland
Paper
This report is printed on BVS, a paper produced  
from well-managed forests and other controlled sources  
certified by the Forest Stewardship Council (FSC).


In [6]:

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(documents)

print("Total pages:", len(documents))
print("Total chunks:", len(chunks))

if chunks:
    print("\nFirst chunk:")
    print(chunks[0].page_content)
    print("\nMetadata:")
    print(chunks[0].metadata)


Total pages: 8
Total chunks: 24

First chunk:
Policy
Mandatory
September  2012
The Nestlé  
Human Resources Policy

Metadata:
{'producer': 'Adobe PDF Library 10.0.1', 'creator': 'Adobe InDesign CS6 (Macintosh)', 'creationdate': '2013-02-12T08:06:14+01:00', 'moddate': '2013-10-31T10:20:17+01:00', 'trapped': '/False', 'source': '/content/the_nestle_hr_policy_pdf_2012.pdf', 'total_pages': 8, 'page': 0, 'page_label': '1'}


In [7]:
import os
from getpass import getpass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass(
        "Enter your OpenAI API key: "
    )

print("API key is configured.")


Enter your OpenAI API key: ··········
API key is configured.


In [8]:

import os
from getpass import getpass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass(
        "Enter your OpenAI API key: "
    )

print("API key is configured.")


API key is configured.


In [9]:

import os

print("API key configured:", bool(os.environ.get("OPENAI_API_KEY")))


API key configured: True


In [10]:

from openai import OpenAI

client = OpenAI(
    api_key=os.environ["OPENAI_API_KEY"],
    timeout=20.0,
    max_retries=0
)

try:
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input="Testing my Nestlé HR Assistant project."
    )
    print("OpenAI connection successful!")
    print("Embedding dimensions:", len(response.data[0].embedding))
except Exception as e:
    print(type(e).__name__, str(e))


RateLimitError Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}


In [11]:

%pip install -q sentence-transformers langchain-huggingface langchain-chroma


In [12]:

from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

print("Local embedding model loaded!")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Local embedding model loaded!


In [13]:

from langchain_chroma import Chroma

if not chunks:
    raise ValueError("No text chunks found. Reload and split the PDF first.")

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="nestle_hr_policies"
)

print("ChromaDB created successfully!")
print("Chunks indexed:", len(chunks))




ChromaDB created successfully!
Chunks indexed: 24


In [14]:

question = "What is the purpose of the HR policy?"

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

results = retriever.invoke(question)

for i, doc in enumerate(results, start=1):
    print(f"\n--- Retrieved passage {i} ---")
    print(doc.page_content)
    print("Page:", doc.metadata.get("page"))



--- Retrieved passage 1 ---
Therefore, managers are committed to build 
and sustain, with their teams, an environment 
of mutual trust. HR ensures that a respectful 
dialogue is present and the voice of the 
employees is heard.
Corporate policy: 
Policy on Conditions of Work and Employment
 Employment and working conditions
Page: 4

--- Retrieved passage 2 ---
Policy
Mandatory
September  2012
The Nestlé  
Human Resources Policy
Page: 0

--- Retrieved passage 3 ---
in terms of remuneration and benefits, if needed 
with the support of HR management, in order to 
communicate properly, clearly and with sufficient 
transparency.
Corporate policy: 
Nestlé Total Rewards Policy
We are committed to providing our employees all 
over the world with good working conditions, a 
safe and healthy work environment, and flexible 
employment possibilities that support a better 
balance of private and professional life consistent 
with our ambition as a leading Nutrition, Health 
and Wellness Company.
A

In [15]:

%pip install -q sentence-transformers langchain-huggingface langchain-chroma


In [16]:

question = "What is the purpose of the HR policy?"

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

results = retriever.invoke(question)

for i, doc in enumerate(results, start=1):
    print(f"\n--- Retrieved passage {i} ---")
    print(doc.page_content)
    print("Page:", doc.metadata.get("page"))



--- Retrieved passage 1 ---
Therefore, managers are committed to build 
and sustain, with their teams, an environment 
of mutual trust. HR ensures that a respectful 
dialogue is present and the voice of the 
employees is heard.
Corporate policy: 
Policy on Conditions of Work and Employment
 Employment and working conditions
Page: 4

--- Retrieved passage 2 ---
Policy
Mandatory
September  2012
The Nestlé  
Human Resources Policy
Page: 0

--- Retrieved passage 3 ---
in terms of remuneration and benefits, if needed 
with the support of HR management, in order to 
communicate properly, clearly and with sufficient 
transparency.
Corporate policy: 
Nestlé Total Rewards Policy
We are committed to providing our employees all 
over the world with good working conditions, a 
safe and healthy work environment, and flexible 
employment possibilities that support a better 
balance of private and professional life consistent 
with our ambition as a leading Nutrition, Health 
and Wellness Company.
A

In [17]:

%pip install -q transformers sentencepiece


In [19]:
import torch
from transformers import AutoTokenizer, T5ForConditionalGeneration

model_name = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)
model.eval()

print("FLAN-T5 model loaded successfully!")



tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

FLAN-T5 model loaded successfully!


In [23]:
print(answer_question("What is the purpose of the HR policy?"))

to communicate properly, clearly and with sufficient transparency

Source pages: 1, 5


In [28]:
%pip install -q gradio

In [29]:
import gradio as gr

demo = gr.Interface(
    fn=answer_question,
    inputs=gr.Textbox(
        lines=2,
        placeholder="Ask a question about Nestlé's HR policy...",
        label="Your question"
    ),
    outputs=gr.Textbox(
        lines=6,
        label="HR Assistant Answer"
    ),
    title="Nestlé AI-Powered HR Assistant",
    description=(
        "Ask questions about the uploaded HR policy. "
        "Answers are generated using retrieved document passages "
        "and a local language model."
    )
)

demo.launch(share=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>

In [30]:
demo.launch(share=True)

Rerunning server... use `close()` to stop if you need to change `launch()` parameters.
----
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3650c8322319d6e6c5.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [22]:
def answer_question(question):
    # 1. Retrieve relevant passages from ChromaDB
    retrieved_docs = retriever.invoke(question)

    if not retrieved_docs:
        return "I couldn't find relevant information in the HR policy."

    # 2. Combine the retrieved passages
    context = "\n\n".join(
        doc.page_content for doc in retrieved_docs
    )

    # 3. Create the prompt BEFORE using it
    prompt = f"""
Answer the question using only the HR policy context below.
If the answer is not present, say the context does not provide
enough information. Do not invent HR policies.

Context:
{context}

Question: {question}

Answer:
"""

    # 4. Tokenize the prompt
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    # 5. Generate an answer using FLAN-T5
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=False
        )

    answer = tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    )

    # 6. Include source page numbers when available
    pages = sorted({
        doc.metadata.get("page", -1) + 1
        for doc in retrieved_docs
        if isinstance(doc.metadata.get("page"), int)
        and doc.metadata.get("page", -1) >= 0
    })

    if pages:
        answer += "\n\nSource pages: " + ", ".join(
            map(str, pages)
        )

    return answer


In [25]:

from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

print("Embedding model initialized successfully!")


Embedding model initialized successfully!


In [26]:

from google.colab import files
from langchain_community.document_loaders import PyPDFLoader
from pathlib import Path

# Find the PDF uploaded to Colab
pdf_files = list(Path("/content").glob("*.pdf"))

if not pdf_files:
    raise FileNotFoundError(
        "No PDF found. Please upload your HR policy PDF."
    )

# Select the uploaded policy PDF
pdf_path = pdf_files[0]

# Load the PDF with LangChain
loader = PyPDFLoader(str(pdf_path))
documents = loader.load()

print("PDF loaded successfully!")
print("File name:", pdf_path.name)
print("Total pages:", len(documents))


PDF loaded successfully!
File name: the_nestle_hr_policy_pdf_2012.pdf
Total pages: 8


In [31]:

for i, doc in enumerate(documents[:2]):
    print(f"\n--- Page {i + 1} ---")
    print(doc.page_content[:1500])



--- Page 1 ---
Policy
Mandatory
September  2012
The Nestlé  
Human Resources Policy

--- Page 2 ---
Policy
Mandatory
September  20
12
Issuing departement
Hum
an Resources
T arget audience 
All
 employees
Approver
Executive Board, Nestlé S.A.
Repository
All Nestlé Principles and Policies, Standards and  
Guidelines can be found in the Centre online repository at:  
http://intranet.nestle.com/nestledocs
Copyright and confidentiality
Al
l rights belong to Nestec Ltd., Vevey, Switzerland.
© 2012, Nestec Ltd.
Design
Nestec Ltd., Corporate Identity & Design,  
Vevey, Switzerland
Production
brain’print GmbH, Switzerland
Paper
This report is printed on BVS, a paper produced  
from well-managed forests and other controlled sources  
certified by the Forest Stewardship Council (FSC).


In [32]:

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(documents)

print("Total pages:", len(documents))
print("Total text chunks:", len(chunks))

if chunks:
    print("\nExample chunk:")
    print(chunks[0].page_content)
    print("\nMetadata:")
    print(chunks[0].metadata)
else:
    print("No text extracted. The PDF may need OCR.")


Total pages: 8
Total text chunks: 24

Example chunk:
Policy
Mandatory
September  2012
The Nestlé  
Human Resources Policy

Metadata:
{'producer': 'Adobe PDF Library 10.0.1', 'creator': 'Adobe InDesign CS6 (Macintosh)', 'creationdate': '2013-02-12T08:06:14+01:00', 'moddate': '2013-10-31T10:20:17+01:00', 'trapped': '/False', 'source': '/content/the_nestle_hr_policy_pdf_2012.pdf', 'total_pages': 8, 'page': 0, 'page_label': '1'}


In [33]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(documents)

print("Total pages:", len(documents))
print("Total text chunks:", len(chunks))

if chunks:
    print("\nExample chunk:")
    print(chunks[1].page_content)
    print("\nMetadata:")
    print(chunks[1].metadata)
else:
    print("No text extracted. The PDF may need OCR.")

Total pages: 8
Total text chunks: 24

Example chunk:
Policy
Mandatory
September  20
12
Issuing departement
Hum
an Resources
T arget audience 
All
 employees
Approver
Executive Board, Nestlé S.A.
Repository
All Nestlé Principles and Policies, Standards and  
Guidelines can be found in the Centre online repository at:  
http://intranet.nestle.com/nestledocs
Copyright and confidentiality
Al
l rights belong to Nestec Ltd., Vevey, Switzerland.
© 2012, Nestec Ltd.
Design
Nestec Ltd., Corporate Identity & Design,  
Vevey, Switzerland
Production
brain’print GmbH, Switzerland
Paper
This report is printed on BVS, a paper produced  
from well-managed forests and other controlled sources  
certified by the Forest Stewardship Council (FSC).

Metadata:
{'producer': 'Adobe PDF Library 10.0.1', 'creator': 'Adobe InDesign CS6 (Macintosh)', 'creationdate': '2013-02-12T08:06:14+01:00', 'moddate': '2013-10-31T10:20:17+01:00', 'trapped': '/False', 'source': '/content/the_nestle_hr_policy_pdf_2012.pdf', 'to

In [34]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(documents)

print("Total pages:", len(documents))
print("Total text chunks:", len(chunks))

if chunks:
    print("\nExample chunk:")
    print(chunks[2].page_content)
    print("\nMetadata:")
    print(chunks[2].metadata)
else:
    print("No text extracted. The PDF may need OCR.")

Total pages: 8
Total text chunks: 24

Example chunk:
The Nestlé Human Resources Policy
1
At Nestlé, we recognize that our employees 
are the key to our success and nothing can be 
achieved without their engagement. 
This document encompasses the guidelines 
which constitute a solid basis for effective Human 
Resources Management throughout the Nestlé 
Group around the world. It explains to all Nestlé 
employees the vision and mission of the Human 
Resources function and illustrates every aspect of 
the Nestlé employee lifecycle. 
The Nestlé Management and Leadership 
Principles inspire all the Nestlé employees in their 
actions and in their dealings with others. The 
Corporate Business Principles refer to all the basic 
principles which Nestlé endorses and subscribes 
to on a worldwide basis. Both these documents

Metadata:
{'producer': 'Adobe PDF Library 10.0.1', 'creator': 'Adobe InDesign CS6 (Macintosh)', 'creationdate': '2013-02-12T08:06:14+01:00', 'moddate': '2013-10-31T10:20:17+0

In [35]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(documents)

print("Total pages:", len(documents))
print("Total text chunks:", len(chunks))

if chunks:
    print("\nExample chunk:")
    print(chunks[3].page_content)
    print("\nMetadata:")
    print(chunks[3].metadata)
else:
    print("No text extracted. The PDF may need OCR.")

Total pages: 8
Total text chunks: 24

Example chunk:
principles which Nestlé endorses and subscribes 
to on a worldwide basis. Both these documents 
are the pillars on which the present policy has 
been built.
The implementation of this policy will be 
inspired by sound judgement, compliance with 
local market laws and common sense, taking into 
account the specific context. Its spirit should be 
respected under all circumstances and could be 
summarised in one sentence: At Nestlé we put 
people at the centre of everything we do.
Jean-Marc Duvoisin
Deputy Executive Vice President
 Introduction

Metadata:
{'producer': 'Adobe PDF Library 10.0.1', 'creator': 'Adobe InDesign CS6 (Macintosh)', 'creationdate': '2013-02-12T08:06:14+01:00', 'moddate': '2013-10-31T10:20:17+01:00', 'trapped': '/False', 'source': '/content/the_nestle_hr_policy_pdf_2012.pdf', 'total_pages': 8, 'page': 2, 'page_label': '3'}


In [37]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(documents)

print("Total pages:", len(documents))
print("Total text chunks:", len(chunks))

if chunks:
    print("\nExample chunk:")
    print(chunks[4].page_content)
    print("\nMetadata:")
    print(chunks[4].metadata)
else:
    print("No text extracted. The PDF may need OCR.")

Total pages: 8
Total text chunks: 24

Example chunk:
The Nestlé Human Resources Policy
2
Line managers have the prime responsibility for 
building and sustaining an environment where 
people have a sense of personal commitment 
to their work and give their best to ensure our 
Company’s success. They care for and develop 
the leaders of tomorrow.
Line managers decide on all people matters 
under their influence, within the boundaries set 
by the policies and principles, acting as the final 
decision makers.
The Human Resources (HR) structure 
enables and empowers them in establishing 
business needs and their corresponding people 
requirements. 
Therefore, the mission of HR managers and 
their teams is to provide professional guidance 
to line managers aiming to deliver superior 
business results by optimising the performance

Metadata:
{'producer': 'Adobe PDF Library 10.0.1', 'creator': 'Adobe InDesign CS6 (Macintosh)', 'creationdate': '2013-02-12T08:06:14+01:00', 'moddate': '2013-10-3

In [36]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(documents)

print("Total pages:", len(documents))
print("Total text chunks:", len(chunks))

if chunks:
    print("\nExample chunk:")
    print(chunks[7].page_content)
    print("\nMetadata:")
    print(chunks[7].metadata)
else:
    print("No text extracted. The PDF may need OCR.")

Total pages: 8
Total text chunks: 24

Example chunk:
candidate remains in the hands of the responsible 
manager, supported by the HR team. 
 Joining Nestlé

Metadata:
{'producer': 'Adobe PDF Library 10.0.1', 'creator': 'Adobe InDesign CS6 (Macintosh)', 'creationdate': '2013-02-12T08:06:14+01:00', 'moddate': '2013-10-31T10:20:17+01:00', 'trapped': '/False', 'source': '/content/the_nestle_hr_policy_pdf_2012.pdf', 'total_pages': 8, 'page': 3, 'page_label': '4'}
